## 9.5 文書検索モデルとChatGPTを組み合わせる

### 9.5.1 検索モデルの準備

In [1]:
!pip install 'datasets<4.0.0' openai==0.27 tiktoken 'transformers[ja]<4.41.0'  faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 17.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 127.3 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 40.7 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 87.7 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 99.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from datasets import load_dataset
from transformers import pipeline

dataset_name = "llm-book/aio-passages-bpr-bert-base-japanese-v3"
passage_dataset = load_dataset(dataset_name, split="train")

encoder_model_name = "llm-book/bert-base-japanese-v3-bpr-question-aio"
encoder_pipeline = pipeline(
    "feature-extraction", model=encoder_model_name, device="cuda:0"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00007-511e49a90d2afb(…):   0%|          | 0.00/343M [00:00<?, ?B/s]

data/train-00001-of-00007-6c92cf6fc0e424(…):   0%|          | 0.00/328M [00:00<?, ?B/s]

data/train-00002-of-00007-0dbf15822023e3(…):   0%|          | 0.00/307M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/634 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [3]:
print(passage_dataset)

Dataset({
    features: ['id', 'pageid', 'revid', 'text', 'section', 'title', 'embeddings'],
    num_rows: 4288198
})


In [4]:
print(passage_dataset[0]["embeddings"])

[133, 162, 145, 21, 151, 215, 254, 119, 214, 80, 4, 189, 177, 53, 100, 115, 68, 177, 70, 103, 41, 90, 127, 227, 113, 27, 71, 148, 92, 162, 176, 133, 105, 99, 16, 16, 52, 70, 132, 46, 2, 32, 211, 149, 29, 103, 53, 233, 29, 199, 124, 65, 178, 90, 60, 32, 201, 114, 214, 132, 60, 254, 216, 249, 184, 57, 119, 181, 23, 253, 121, 83, 63, 115, 141, 73, 90, 0, 239, 225, 194, 248, 16, 108, 66, 215, 124, 248, 0, 26, 214, 78, 52, 118, 115, 194]


In [5]:
print(dir(passage_dataset))

['_TF_DATASET_REFS', '__class__', '__del__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getitems__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_build_local_temp_path', '_check_index_is_initialized', '_data', '_estimate_nbytes', '_fingerprint', '_format_columns', '_format_kwargs', '_format_type', '_generate_tables_from_cache_file', '_generate_tables_from_shards', '_get_cache_file_path', '_get_output_signature', '_getitem', '_indexes', '_indices', '_info', '_map_single', '_new_dataset_with_indices', '_output_all_columns', '_push_parquet_shards_to_hub', '_save_to_disk_single', '_select_contiguous', '_select_with_indices_mapping', '_split', 'add_colum

#### `passage_dataset["embeddings"]`を`Faiss`ライブラリのインデックスに格納する

In [6]:
import faiss
import numpy as np
from tqdm import tqdm

# インデックスの初期化
embed_size = encoder_pipeline.model.config.hidden_size
faiss_index = faiss.IndexBinaryIDMap2(
    faiss.IndexBinaryFlat(embed_size)
)

with tqdm(total=len(passage_dataset)) as pbar:
    i = 0
    for batch in passage_dataset.iter(batch_size=512):
        bs = len(batch["embeddings"])

        # 埋め込みをインデックスに適したdtypeへと変換
        batch_embeddings = np.array(
            batch["embeddings"], dtype=np.uint8
        )
        batch_indices = np.arange(i, i + bs, dtype=np.int64)

        # 埋め込みをインデックスに格納
        faiss_index.add_with_ids(batch_embeddings, batch_indices)

        pbar.update(n=bs)
        i += bs

100%|██████████| 4288198/4288198 [03:43<00:00, 19190.86it/s]


In [7]:
import torch

def embed_questions(questions: list[str]) -> np.ndarray:
    """質問文を実数ベクトルに変換"""
    output_tensors = encoder_pipeline(questions, return_tensors="pt")
    embeddings = np.stack([
        t.squeeze(0)[0].numpy().astype(np.float32)
        for t in output_tensors
    ])
    return embeddings

def binarize_embeddings(embeddings: np.ndarray) -> np.ndarray:
    """実数ベクトルをバイナリベクトルに変換"""
    # 0未満の値を0に、0以上の値を1に変換
    binary_embeddings = np.where(embeddings < 0, 0, 1)
    # バイト単位でまとめてuint8に変換
    packed_binary_embeddings = np.packbits(binary_embeddings, axis=1)
    return packed_binary_embeddings

In [8]:
q_embed = embed_questions(["日本で一番高い山は何？"])
binary_q_embed = binarize_embeddings(q_embed)
print(binary_q_embed)

[[141 210 243   2 153 138 188 209 212 205 166 253 146  26  12 135 215 208
    0 233 223  17  43   5 198 163   1 116 111  63 138 212 106 118 126  84
   31 166  23 194 190 164 201 165 139  83 132  46  86  11  97  24  81 227
   12  46 128 115  17  49  22 224 101  16 252 100 229  33  92  77 216 119
  176  62 252  97  16 110 203  36 195 208  32 170 138 159 255  83  83 122
   64 166 100 246  83  44]]


In [11]:
scores, passage_ids = faiss_index.search(binary_q_embed, k=5)
print(f"scores: {scores} \n passageId: {passage_ids}")

scores: [[168 169 170 173 174]] 
 passageId: [[2222502 2783191 2879647 3581406 1149955]]


In [12]:
print(passage_dataset[passage_ids[0]]["text"])

['日本の領土に占める山間部の割合はおよそ7割程度である。2020年現在、日本で一番高い山は富士山で、逆に一番低い山は日和山である。日本は新期造山帯に位置し、多くの火山が見られる。日本では昔から人々が山と共生する文化が培われてきた。それらの山は里山とよばれ、農村などでは山を共有地として村全体で管理し、薪をとったり土や山菜などを利用する目的で利用され、手入れをされてきた。しかし、明治時代になるとそれらの山の中には国有地にされてしまったものも多く存在する。また、日本の山の中には霊峰とよばれ、民間信仰の場にされた山も多くある。山の中には昔から金山や銀山として、近代では銅や石炭などを入手するため多くの鉱山が作られ、鉱毒や粉塵などが問題になった。特に近代において、山林の多くで木材を得るための大規模な伐採などが行われた。', '3000 m峰は、独立峰の富士山と御嶽山及び飛騨山脈(北アルプス)と赤石山脈(南アルプス)の山域に限られている。24番目に高い山は剱岳 (2999 m) で、3000 mにわずか1 m届いていない。標高3000 mを越える一帯は森林限界のハイマツ帯で、高山植物の群生地となっている。また多くの3000 m峰のハイマツ帯は、ライチョウ(雷鳥)の生息地となっている。', '三角点は一般的に眺望の利く場所に設置され一等三角点は970点余りに上るが、その中で風格のある山容、優れた眺望、高い知名度、さらに概ね標高1,000m以上で登りがいのある山が選定された。一等三角点の最高峰は、標高3,121 mの南アルプスの赤石岳である。三角点がその山の最高峰とは限らず、山頂に三角点より高い場所があり、その地点すなわち標高点や測定点を以て山の高さとしている例も少なくない(例:早池峰、御嶽山、金剛山)。また連山を形成している場合、最高峰でないピークに一等三角点が設置されている例もある(例:赤城山、穂高岳、阿蘇山)。特殊な例としては雲仙普賢岳のように、噴火により一等三角点が最高峰でなくなった事例もある。利尻岳中腹の長官山のように全く山頂とは離れた地点に一等三角点が設置されている例もある(山頂は二等三角点「利尻絶頂」)。', 'この項では世界最高峰と考えられていた山(せかいさいこうほうとかんがえられていたやま)について述べる。19世紀初頭までアンデスが世界で最も高い山脈だと考えられて

In [13]:
passage_id = 0
print(np.unpackbits(faiss_index.reconstruct(passage_id)))

[1 0 0 0 0 1 0 1 1 0 1 0 0 0 1 0 1 0 0 1 0 0 0 1 0 0 0 1 0 1 0 1 1 0 0 1 0
 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 0 0 1 1 1 0 1 1 1 1 1 0 1 0 1 1 0 0 1
 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 1 1 0 1 1 0 1 1 0 0 0 1 0 0 1 1 0 1 0
 1 0 1 1 0 0 1 0 0 0 1 1 1 0 0 1 1 0 1 0 0 0 1 0 0 1 0 1 1 0 0 0 1 0 1 0 0
 0 1 1 0 0 1 1 0 0 1 1 1 0 0 1 0 1 0 0 1 0 1 0 1 1 0 1 0 0 1 1 1 1 1 1 1 1
 1 1 0 0 0 1 1 0 1 1 1 0 0 0 1 0 0 0 1 1 0 1 1 0 1 0 0 0 1 1 1 1 0 0 1 0 1
 0 0 0 1 0 1 1 1 0 0 1 0 1 0 0 0 1 0 1 0 1 1 0 0 0 0 1 0 0 0 0 1 0 1 0 1 1
 0 1 0 0 1 0 1 1 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 1 0 0
 0 1 0 0 0 1 1 0 1 0 0 0 0 1 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0
 0 0 0 1 1 0 1 0 0 1 1 1 0 0 1 0 1 0 1 0 0 0 1 1 1 0 1 0 1 1 0 0 1 1 1 0 0
 1 1 0 1 0 1 1 1 1 0 1 0 0 1 0 0 0 1 1 1 0 1 1 1 0 0 0 1 1 1 0 1 1 1 1 1 0
 0 0 1 0 0 0 0 0 1 1 0 1 1 0 0 1 0 0 1 0 1 1 0 1 0 0 0 1 1 1 1 0 0 0 0 1 0
 0 0 0 0 1 1 0 0 1 0 0 1 0 1 1 1 0 0 1 0 1 1 0 1 0 1 1 0 1 0 0 0 0 1 0 0 0
 0 1 1 1 1 0 0 1 1 1 1 1 

In [14]:
import numpy as np
import torch

def retrieve_passages(
    questions: list[str],
    binary_k: int=2048,  # リランキングするための候補の取得数
    top_k: int=3  # リランキング後に出力する最終的なパッセージ数
) -> list[list[str]]:
    """質問をバイナリベクトルに変換する"""
    q_embed = embed_questions(questions)
    q_bin_embed = binarize_embeddings(q_embed)

    # バイナリベクトルを使用して検索する
    binary_k_scores, binary_k_p_ids = faiss_index.search(
        q_bin_embed, binary_k
    )

    batch_size = len(questions)
    embed_dim = q_bin_embed.shape[-1] * 8
    # インデックスからパッセージのベクトルを復元
    p_uint8_embed = [
        faiss_index.reconstruct(int(p_id))
        for p_id in binary_k_p_ids.flatten()
    ]
    # uint8からboolに変換
    p_bin_embed = np.vstack([np.unpackbits(e) for e in p_uint8_embed])
    # 型とshapeを変換
    p_bin_embed = p_bin_embed.astype(np.float32).reshape(
        batch_size, binary_k, embed_dim
    )
    p_bin_embed = p_bin_embed * 2 - 1

    # 質問の実数ベクトルとパッセージのバイナリベクトルで再度スコアを計算する
    re_scores = np.einsum("ijk,ik->ij", p_bin_embed, q_embed)
    top_k_indices = np.argsort(-re_scores, axis=-1)[:, :top_k]
    top_k_p_ids = np.take_along_axis(
        binary_k_p_ids, top_k_indices, axis=-1
    )

    # top_kのテキストを整形して出力する
    retrieved_texts: list[list[str]] = []
    for p_ids in top_k_p_ids:
        formatted_texts = [
            f"タイトル: {passage_dataset[i]['title']}\n"
            f"本文: {passage_dataset[i]['text']}"
            for i in p_ids.tolist()
        ]
        retrieved_texts.append(formatted_texts)
    
    return retrieved_texts

In [19]:
passages = retrieve_passages(["神奈川県の県庁所在地は？"], top_k=3)[0]
for passage in passages:
    print(passage)

タイトル: 神奈川県
本文: 藤沢市は人口が30万人を超えているが(約43万人)、中核市に指定されていない。県東部の横浜市、川崎市は、都市化、工業化が進んでおり、東京湾に面した京浜工業地帯の一角を形成する。県西部は緑豊かな丹沢山地から足柄山地、箱根山が連なり、酒匂川が流れる足柄平野には小田原城の城下町・小田原市が開ける。県中央部は相模原市、厚木市、海老名市などの平野部で都市化・工業化が進んでおり、相模川が流れ平塚市から相模湾に注ぐ。県南東部は、海沿いに茅ヶ崎市、藤沢市が開け、鎌倉幕府が置かれた鎌倉市から、明治以来の軍港都市・横須賀市がある三浦半島にかけて、三浦丘陵が連なる。新都心の横浜みなとみらい21、横浜赤レンガ倉庫、横浜中華街といった都市部の観光地だけでなく、箱根、熱海、茅ヶ崎、江の島といった山や海のリゾート地にも恵まれている事から日本全国から観光客が訪れる県になっている。
タイトル: 神奈川県
本文: 神奈川県(かながわけん、(英: Kanagawa Prefecture)は、日本の関東地方に位置する県。県庁所在地は横浜市。
タイトル: 神奈川県私立中学高等学校協会
本文: 横浜市(神奈川区・中区・西区・保土ケ谷区 ・南区)横浜市(港北区・鶴見区・都筑区)横浜市(泉区・磯子区・金沢区・港南区・栄区・戸塚区)横浜市(青葉区・旭区・瀬谷区・緑区)川崎市(麻生区・川崎区・幸区・高津区・多摩区・中原区・宮前区)鎌倉市・逗子市・横須賀市・三浦市・三浦郡(葉山町)綾瀬市・厚木市・伊勢原市・海老名市・相模原市・座間市・秦野市・大和市・愛甲郡(愛川町・清川村)・高座郡(寒川町)茅ヶ崎市・平塚市・藤沢市・中郡(大磯町)小田原市・南足柄市・足柄上郡(大井町・開成町・中井町・松田町・山北町)・足柄下郡(箱根町・真鶴町・湯河原町)・中郡(二宮町)


### 9.5.2 検索結果のプロンプトへの組み込み

In [20]:
import os
os.environ["OPENAI_API_KEY"] = 

SyntaxError: invalid syntax (4112619785.py, line 2)

In [21]:
from abc import ABCMeta, abstractmethod

class PromptMaker(metaclass=ABCMeta):
    """クイズ用プロンプトを作成するための抽象クラス"""

    @abstractmethod
    def run(self, questions: list[str]) -> list[str]:
        """プロンプトの作成（具体的な実装は継承先で行われる）"""
        pass

class RetrievalPromptMaker(PromptMaker):
    """
    質問から関連するテキストを取得し、
    それを組み入れたプロンプトを作成する
    """
    def __init__(self, binary_k: int=2048, top_k: int=3):
        """初期化処理の定義"""
        self.binary_k = binary_k
        self.top_k = top_k
    
    def run(self, questions: list[str]) -> list[str]:
        """プロンプトを作成"""
        retrieved_passages_list = retrieve_passages(
            questions, binary_k=self.binary_k, top_k=self.top_k
        )
        retrieved_texts = [
            "\n".join(ps) for ps in retrieved_passages_list
        ]
        return [
            "あなたには今からクイズに答えてもらいます。"
            "問題を与えますので、その解答のみを簡潔に出力してください。\n"
            "また解答の参考になりうるテキストを与えます。"
            "解答を含まない場合もあるのでその場合は無視してください。\n"
            "\n"
            f"{text}\n"
            "\n"
            f"問題：{question}\n"
            "解答："
            for question, text in zip(questions, retrieved_texts)
        ]

In [22]:
retrieval_prompt_maker = RetrievalPromptMaker(top_k=3)
print(retrieval_prompt_maker.run(["日本で一番高い山は何？"])[0])

あなたには今からクイズに答えてもらいます。問題を与えますので、その解答のみを簡潔に出力してください。
また解答の参考になりうるテキストを与えます。解答を含まない場合もあるのでその場合は無視してください。

タイトル: 日本の山
本文: 日本の領土に占める山間部の割合はおよそ7割程度である。2020年現在、日本で一番高い山は富士山で、逆に一番低い山は日和山である。日本は新期造山帯に位置し、多くの火山が見られる。日本では昔から人々が山と共生する文化が培われてきた。それらの山は里山とよばれ、農村などでは山を共有地として村全体で管理し、薪をとったり土や山菜などを利用する目的で利用され、手入れをされてきた。しかし、明治時代になるとそれらの山の中には国有地にされてしまったものも多く存在する。また、日本の山の中には霊峰とよばれ、民間信仰の場にされた山も多くある。山の中には昔から金山や銀山として、近代では銅や石炭などを入手するため多くの鉱山が作られ、鉱毒や粉塵などが問題になった。特に近代において、山林の多くで木材を得るための大規模な伐採などが行われた。
タイトル: 日本の山一覧 (3000m峰)
本文: 3000 m峰は、独立峰の富士山と御嶽山及び飛騨山脈(北アルプス)と赤石山脈(南アルプス)の山域に限られている。24番目に高い山は剱岳 (2999 m) で、3000 mにわずか1 m届いていない。標高3000 mを越える一帯は森林限界のハイマツ帯で、高山植物の群生地となっている。また多くの3000 m峰のハイマツ帯は、ライチョウ(雷鳥)の生息地となっている。
タイトル: 一等三角点百名山
本文: 三角点は一般的に眺望の利く場所に設置され一等三角点は970点余りに上るが、その中で風格のある山容、優れた眺望、高い知名度、さらに概ね標高1,000m以上で登りがいのある山が選定された。一等三角点の最高峰は、標高3,121 mの南アルプスの赤石岳である。三角点がその山の最高峰とは限らず、山頂に三角点より高い場所があり、その地点すなわち標高点や測定点を以て山の高さとしている例も少なくない(例:早池峰、御嶽山、金剛山)。また連山を形成している場合、最高峰でないピークに一等三角点が設置されている例もある(例:赤城山、穂高岳、阿蘇山)。特殊な例としては雲仙普賢岳のように、噴火により一等三角点が最高